In [ ]:
import os
os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=4"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import jax
import jax.numpy as jnp
import jax.random as random

print("Devices:", jax.devices())
print("Default backend:", jax.default_backend())

import numpyro
import numpyro.distributions as dist
from numpyro import sample
from numpyro.infer import MCMC, NUTS, init_to_median
from numpyro.infer.reparam import TransformReparam
from numpyro.handlers import reparam
from numpyro.distributions import transforms
from numpyro.infer.initialization import init_to_feasible
from numpyro.infer import Predictive

numpyro.set_platform("cpu")

from itertools import combinations
from statsmodels.stats.outliers_influence import variance_inflation_factor
import arviz as az
from scipy import stats


rng_key = jax.random.PRNGKey(1)
num_chains = 4

rope = 0.05
rope_interval = np.array([-rope, rope])

os.makedirs('../plots', exist_ok=True)
os.makedirs('../parameter_estimates', exist_ok=True)

#set number of samples
num_samples = 2000
num_warmup = 2000

In [ ]:
print(jax.local_device_count())

#get GPU if available
device = jax.devices("gpu")[0] if jax.extend.backend.get_backend().platform == "gpu" else "cpu"
print(device)

### Data preparation


In [ ]:
# load data -- analyse the 4 outcomes in separate GLMs
dataset = "misinfo" #misinfo, trust, private, extreme 

if dataset == "misinfo":    
    data = pd.read_csv('../data/Ztable_misinfo_combined.csv')
elif dataset == "trust":
    data = pd.read_csv('../data/Ztable_trust_combined.csv')
elif dataset == "private":
    data = pd.read_csv('../data/Ztable_private_combined.csv')
elif dataset == "extreme":
    data = pd.read_csv('../data/Ztable_extreme_combined.csv')


print(len(data))
print(data["model"].unique())
data.head(10)

In [ ]:
# Create model_subject identifier
data['model_subject'] = data['model'] + '_' + data['subject'].astype(str)

# Get one row per subject (since sub_compliance is constant within subject)
subject_compliance = data.groupby(['model_subject', 'model', 'iscontrol'])['sub_compliance'].first().reset_index()

# Mean and STD per model and iscontrol
summary = subject_compliance.groupby(['model', 'iscontrol'])['sub_compliance'].agg(['mean', 'std', 'count']).reset_index()

print(summary)

In [ ]:
# Within each model
print("=" * 70)
print("T-tests within each model (control vs test)")
print("=" * 70)

for model in subject_compliance['model'].unique():
    control = subject_compliance[(subject_compliance['model'] == model) & 
                                 (subject_compliance['iscontrol'] == 'control')]['sub_compliance']
    test = subject_compliance[(subject_compliance['model'] == model) & 
                              (subject_compliance['iscontrol'] == 'test')]['sub_compliance']
    
    t_stat, p_value = stats.ttest_ind(control, test)
    print(f"{model:30s} | t = {t_stat:7.3f} | p = {p_value:.4f} | "
          f"Control M = {control.mean():.3f} | Test M = {test.mean():.3f} | "
          f"Diff = {test.mean() - control.mean():.3f}")

# Aggregated across models
print("\n" + "=" * 70)
print("T-test aggregated across all models (control vs test)")
print("=" * 70)

all_control = subject_compliance[subject_compliance['iscontrol'] == 'control']['sub_compliance']
all_test = subject_compliance[subject_compliance['iscontrol'] == 'test']['sub_compliance']

t_stat, p_value = stats.ttest_ind(all_control, all_test)
print(f"{'Aggregated':30s} | t = {t_stat:7.3f} | p = {p_value:.4f} | "
      f"Control M = {all_control.mean():.3f} | Test M = {all_test.mean():.3f} | "
      f"Diff = {all_test.mean() - all_control.mean():.3f}")


In [ ]:
unique_combos = data[['question', 'topic', 'ground_truth']].value_counts().reset_index().sort_values(['topic', 'count'], ascending=[True, False])

unique_combos

In [ ]:
#find number of unique subjects in data["subject"]
num_subjects = len(data["subject"].unique())
print(num_subjects)

#split number of unique subjects in data["subject"] by iscontrol
num_control = len(data[data["iscontrol"] == 'control']["subject"].unique())
num_test = len(data[data["iscontrol"] == 'test']["subject"].unique())

print(num_control)
print(num_test)



#only show length of data for "model" == 'GPT4o' 'mistral' 'claude' 
models = ['GPT4o', 'mistral', 'claude']

data_models = data[data["model"].isin(models)]
num_subjects = len(data_models["subject"].unique())
print(num_subjects)
#split number of unique subjects in data["subject"] by iscontrol
num_control = len(data_models[data_models["iscontrol"] == 'control']["subject"].unique())
num_test = len(data_models[data_models["iscontrol"] == 'test']["subject"].unique())

print(num_control)
print(num_test)


#only show length of data for "model" == 'GPT4o_sycophancy_both' 'GPT4o_persuasion'
models = ['GPT4o_sycophancy_both', 'GPT4o_persuasion']

data_models = data[data["model"].isin(models)]
num_subjects = len(data_models["subject"].unique())
print(num_subjects)
#split number of unique subjects in data["subject"] by iscontrol
num_control = len(data_models[data_models["iscontrol"] == 'control']["subject"].unique())
num_test = len(data_models[data_models["iscontrol"] == 'test']["subject"].unique())

print(num_control)
print(num_test)


In [ ]:
#prepare dataset functions

#generalized function to prepare data for different models
def prepare_data(df, subset_fraction=0.01, use_subset=True, interaction_level=4, dataset="misinfo", exclude_models=True, prompting=0):
    print("\nDistribution in raw data:")
    print(f"ground_truth values: {df['ground_truth'].value_counts().to_dict()}")
    print(f"researched values: {df['researched'].value_counts().to_dict()}")
    print(f"iscontrol values: {df['iscontrol'].value_counts().to_dict()}")

    if prompting == 0:
        df = df[~df['model'].isin(['GPT4o_sycophancy_both', 'GPT4o_persuasion'])].copy()
            
    elif prompting == 1:
        df = df[df['model'].isin(['GPT4o_sycophancy_both', 'GPT4o_persuasion'])].copy()

    # Create time variable (-0.5 for pre, 0.5 for post)
    df['time'] = (df['presented'] == 'post').astype(float) - 0.5

    # Create the numeric versions of categorical variables
    if dataset == "private":
        df['ground_truth_num'] = (df['ground_truth'] == True).astype(float) - 0.5
        df['researched_num'] = (df['researched'] == 'yes').astype(float) - 0.5
        df['istest_num'] = (df['iscontrol'] == 0).astype(float) - 0.5
    else:
        df['ground_truth_num'] = (df['ground_truth'] == True).astype(float) - 0.5
        df['researched_num'] = (df['researched'] == 'yes').astype(float) - 0.5
        df['istest_num'] = (df['iscontrol'] == 'test').astype(float) - 0.5

    if use_subset:
        # Enforce balanced selection by ground_truth
        true_subjects = df[df['ground_truth'] == True]['subject'].unique()
        false_subjects = df[df['ground_truth'] == False]['subject'].unique()
        
        print(f"Total unique subjects: {len(df['subject'].unique())}")
        print(f"Subjects with true ground_truth: {len(true_subjects)}")
        print(f"Subjects with false ground_truth: {len(false_subjects)}")
        
        # Calculate how many subjects to sample from each group
        subject_fraction = subset_fraction
        true_sample_size = max(int(len(true_subjects) * subject_fraction), 3)
        false_sample_size = max(int(len(false_subjects) * subject_fraction), 3)
        
        # Ensure we don't try to sample more than available
        true_sample_size = min(true_sample_size, len(true_subjects))
        false_sample_size = min(false_sample_size, len(false_subjects))
        
        # Sample from each group
        sampled_true_subjects = np.random.choice(true_subjects, true_sample_size, replace=False)
        sampled_false_subjects = np.random.choice(false_subjects, false_sample_size, replace=False)
        
        # Combine the samples
        sampled_subjects = np.concatenate([sampled_true_subjects, sampled_false_subjects])
        
        # Get the subset data
        df = df[df['subject'].isin(sampled_subjects)]
        
        # Check the resulting distribution
        print(f"\nUsing subset of data: {len(df)} observations from {len(sampled_subjects)} subjects")
        print(f"ground_truth distribution in subset: {df['ground_truth'].value_counts().to_dict()}")
        
        # Extra check: verify we have both values in the coded variable
        gt_values = df['ground_truth_num'].unique()
        print(f"ground_truth_num unique values in subset: {gt_values}")

    # Center numerical predictors
    df['sub_compliance_centered'] = (df['sub_compliance'] - df['sub_compliance'].mean()) / df['sub_compliance'].std()

    # Create numeric subject IDs
    # subject_ids = {subj: i for i, subj in enumerate(df['subject'].unique())}
    # df['subject_id'] = df['subject'].map(subject_ids)
    df['model_subject'] = df['model'] + '_' + df['subject'].astype(str)
    subject_ids = {subj: i for i, subj in enumerate(df['model_subject'].unique())}
    df['subject_id'] = df['model_subject'].map(subject_ids)


    # Print values for the recoded variables to check for variance
    print("\nRecoded variable statistics:")
    for var in ['ground_truth_num', 'researched_num', 'istest_num']:
        unique_vals = np.unique(df[var])
        print(f"{var}: unique values = {unique_vals}, variance = {df[var].var():.6f}")

    # Define predictor variables
    main_vars = ['ground_truth', 'researched', 'istest']

    # Only include variables with variance in the design matrix
    selected_vars = []
    for var in main_vars:
        var_data = df[f"{var}_num"]
        if var_data.var() > 0:
            selected_vars.append(var)
        else:
            print(f"WARNING: {var} has no variance in the subset - excluding from model")

    # Start with main effects
    X_dict = {'Intercept': np.ones(len(df)), 'time': df['time']}

    # Add variables with variance
    for var in selected_vars:
        X_dict[var] = df[f"{var}_num"]

    # Add sub_compliance
    X_dict['sub_compliance'] = df['sub_compliance_centered']

    if not exclude_models: 
        if prompting == 0:
            # Define effect coding for model categories
            df['model_gpt4o'] = np.where(df['model'] == 'GPT4o', 0.5, np.where(df['model'] == 'claude', -0.25, -0.25))
            df['model_claude'] = np.where(df['model'] == 'claude', 0.5, np.where(df['model'] == 'GPT4o', -0.25, -0.25))
            df['model_mistral'] = np.where(df['model'] == 'mistral', 0.5, np.where(df['model'] == 'GPT4o', -0.25, -0.25))

            # Add effect-coded model columns
            X_dict['model_gpt4o'] = df['model_gpt4o']
            X_dict['model_claude'] = df['model_claude']
            X_dict['model_mistral'] = df['model_mistral']
        elif prompting == 1:
            # Define effect coding for model categories
            df['model_sycophancy'] = np.where(df['model'] == 'GPT4o_sycophancy_both', 0.5, -0.5)
            df['model_persuasion'] = np.where(df['model'] == 'GPT4o_persuasion', 0.5, -0.5)
            
            # Add effect-coded model columns
            X_dict['model_sycophancy'] = df['model_sycophancy']
            X_dict['model_persuasion'] = df['model_persuasion']
    
    X = pd.DataFrame(X_dict)

    # Define a function to add interaction terms
    def add_interactions(X, order, vars_to_interact, model_vars=None):
        """
        Add interactions of a specific order to the design matrix.
        Order is the number of main_vars involved (doesn't include model interactions).
        """
        print(f"\nAdding {order}-way interactions...")

        # Get all combinations of vars_to_interact of the given order
        for var_combo in combinations(vars_to_interact, order):
            # Create the interaction name
            interaction_name = '_'.join(var_combo)

            # Create the base interaction term
            X[interaction_name] = 1.0
            for var in var_combo:
                X[interaction_name] *= X[var]

            # Add interactions with model variables if provided
            if model_vars:
                for model_var in model_vars:
                    model_interaction_name = f"{interaction_name}_{model_var}"
                    X[model_interaction_name] = X[interaction_name] * X[model_var]

        return X

   # Only proceed with interactions if we have variables with variance
    if len(selected_vars) >= 2:
        if not exclude_models:
            if prompting == 0:
                model_vars = ['model_gpt4o', 'model_claude', 'model_mistral']
            elif prompting == 1:
                model_vars = ['model_sycophancy', 'model_persuasion']
        else:
            model_vars = []
                
        # Interactions incrementally by order
        max_order = min(interaction_level, len(selected_vars))
        for order in range(2, max_order + 1):
            X = add_interactions(X, order, selected_vars, model_vars)
        # Add explicit time interaction
        print("\nAdding time interactions...")
        # 2-way: time × each main
        for var in selected_vars:
            X[f"time_{var}"] = X['time'] * X[var]
        # 2-way: time × model if models are included
        if model_vars:
            for model_var in model_vars:
                X[f"time_{model_var}"] = X['time'] * X[model_var]
        # 3-way: time × var × model if models are included
        if model_vars:
            for var in selected_vars:
                for model_var in model_vars:
                    X[f"time_{var}_{model_var}"] = X['time'] * X[var] * X[model_var]
        # Higher-order time interactions
        if interaction_level >= 3 and len(selected_vars) >= 2:
            for order in range(2, min(interaction_level, len(selected_vars)) + 1):
                print(f"Adding {order+1}-way time interactions (time x {order} vars)...")
                for var_combo in combinations(selected_vars, order):
                    interaction_name = 'time_' + '_'.join(var_combo)
                    X[interaction_name] = X['time']
                    for var in var_combo:
                        X[interaction_name] *= X[var]
                    if model_vars:
                        for model_var in model_vars:
                            model_interaction_name = f"{interaction_name}_{model_var}"
                            X[model_interaction_name] = X[interaction_name] * X[model_var]
    else:
        print("WARNING: Not enough variables with variance to create interactions.")

    # Check for numerical issues
    print("\nChecking for numerical issues in design matrix...")
    var_threshold = 1e-10
    low_var_cols = []

    for col in X.columns:
        var_val = X[col].var()
        if var_val < var_threshold and col != 'Intercept':
            low_var_cols.append((col, var_val))

    if low_var_cols:
        print("WARNING: The following columns have very low variance:")
        for col, var_val in low_var_cols:
            print(f"  {col}: {var_val:.10e}")
        # Remove columns with no variance - keep Intercept
        cols_to_keep = [col for col in X.columns if X[col].var() >= var_threshold or col == 'Intercept']
        if cols_to_keep:
            X = X[cols_to_keep]
            print(f"Removed {len(low_var_cols)} columns with no variance.")
        else:
            print("ERROR: No columns with variance left after filtering!")

    # Check for multicollinearity
    try:
        from numpy.linalg import svd
        U, s, Vh = svd(X.values, full_matrices=False)
        condition_number = s[0] / s[-1]
        print(f"Condition number of design matrix: {condition_number:.4e}")

        if condition_number > 1e10:
            print("WARNING: Design matrix is poorly conditioned. Consider simplifying the model.")
    except Exception as e:
        print(f"Could not compute condition number: {e}")

    # Extract response variable
    y = df['response'].values - 1 
    # Reverse code such that 0 is strong disagreement and 6 is strong agreement
    if dataset != "trust":
        y = 6 - (df['response'].values - 1)

    # Print summary statistics
    print("\nResponse variable stats:")
    print(f"  Range: {np.min(y)} to {np.max(y)}")
    print(f"  Mean: {np.mean(y):.4f}, Std: {np.std(y):.4f}")
    print(f"  Unique values: {sorted(np.unique(y))}")

    # Print time effect
    for time_val, time_name in [(-0.5, "Pre"), (0.5, "Post")]:
        time_responses = y[df['time'] == time_val]
        print(f"  {time_name}: n={len(time_responses)}, mean={np.mean(time_responses):.4f}")

    # Subject IDs for random effects
    subject_ids_array = df['subject_id'].values
    num_subjects = len(np.unique(subject_ids_array))
    print(f"\nNumber of subjects: {num_subjects}")

    # Check data per subject
    subject_counts = df.groupby('subject_id').size()
    print(f"Observations per subject: min={subject_counts.min()}, max={subject_counts.max()}, mean={subject_counts.mean():.1f}")

    # Convert to JAX arrays with proper data types
    X_array = jnp.array(X.values, dtype=jnp.float32)

    # If your model is expecting integer outcomes
    y_array = jnp.array(y.astype(np.int32))

    # Subject IDs must be integers
    subject_ids_array = jnp.array(subject_ids_array, dtype=jnp.int32)

    # For ordinal models
    unique_y = np.sort(np.unique(y_array))
    num_cutpoints = len(unique_y) - 1

    print(f"\nFinal design matrix shape: {X_array.shape}")
    print(f"Number of parameters: {X_array.shape[1]}")
    print(f"Number of cutpoints needed for ordinal model: {num_cutpoints}")
    print(f"Response variable type: {y_array.dtype}")
    print(f"Subject IDs type: {subject_ids_array.dtype}")

    return X_array, y_array, subject_ids_array, num_subjects, num_cutpoints, X.columns.tolist(), unique_y



#generalized function for sycophancy and persuasion, prepare data for extremism (sign flip), including or excluding models
def prepare_data_extremism(df, subset_fraction=0.01, use_subset=True, interaction_level=4, exclude_models=True, prompting=0):
    print("\nDistribution in raw data:")
    print(f"ground_truth values: {df['ground_truth'].value_counts().to_dict()}")
    print(f"researched values: {df['researched'].value_counts().to_dict()}")
    print(f"istest values: {df['iscontrol'].value_counts().to_dict()}")

    # Remove specified models
    if prompting == 0:
        df = df[~df['model'].isin(['GPT4o_sycophancy_both', 'GPT4o_persuasion'])].copy()
            
    elif prompting == 1:
        df = df[df['model'].isin(['GPT4o_sycophancy_both', 'GPT4o_persuasion'])].copy()

    # Create the numeric versions of categorical variables
    df['ground_truth_num'] = (df['ground_truth'] == True).astype(float) - 0.5
    df['researched_num'] = (df['researched'] == 'yes').astype(float) - 0.5
    df['istest_num'] = (df['iscontrol'] == 0).astype(float) - 0.5

    if use_subset:
        # Enforce balanced selection by ground_truth
        true_subjects = df[df['ground_truth'] == True]['subject'].unique()
        false_subjects = df[df['ground_truth'] == False]['subject'].unique()
        
        print(f"Total unique subjects: {len(df['subject'].unique())}")
        print(f"Subjects with true ground_truth: {len(true_subjects)}")
        print(f"Subjects with false ground_truth: {len(false_subjects)}")
        
        # Calculate how many subjects to sample from each group
        subject_fraction = subset_fraction
        true_sample_size = max(int(len(true_subjects) * subject_fraction), 3)
        false_sample_size = max(int(len(false_subjects) * subject_fraction), 3)
        
        # Ensure we don't try to sample more than available
        true_sample_size = min(true_sample_size, len(true_subjects))
        false_sample_size = min(false_sample_size, len(false_subjects))
        
        # Sample from each group
        sampled_true_subjects = np.random.choice(true_subjects, true_sample_size, replace=False)
        sampled_false_subjects = np.random.choice(false_subjects, false_sample_size, replace=False)
        
        # Combine the samples
        sampled_subjects = np.concatenate([sampled_true_subjects, sampled_false_subjects])
        
        # Get the subset data
        df = df[df['subject'].isin(sampled_subjects)]
        
        # Check the resulting distribution
        print(f"\nUsing subset of data: {len(df)} observations from {len(sampled_subjects)} subjects")
        print(f"ground_truth distribution in subset: {df['ground_truth'].value_counts().to_dict()}")
        
        # Extra check: verify we have both values in the coded variable
        gt_values = df['ground_truth_num'].unique()
        print(f"ground_truth_num unique values in subset: {gt_values}")

    # Center numerical predictors
    df['sub_compliance_centered'] = (df['sub_compliance'] - df['sub_compliance'].mean()) / df['sub_compliance'].std()

    # Calculate the distance from the center point (4) - scale from 1-7
    center_point = 4
    df['response_centered'] = df['response'] - center_point

    # Pivot the data to get pre and post response on the same row
    pivoted_df = df.pivot_table(index=['subject', 'model', 'topic', 'ground_truth_num', 'researched_num', 'istest_num', 'searchornot'],
                                columns='presented', values='response_centered').reset_index()

    # Calculate if the sign of the response has flipped from pre to post
    pivoted_df['sign_flip'] = (pivoted_df['pre'] * pivoted_df['post'] < 0).astype(int)

    # Merge the sign flip back to the original dataframe
    df = pd.merge(df, pivoted_df[['subject', 'model', 'topic', 'ground_truth_num', 'researched_num', 'istest_num', 'searchornot', 'sign_flip']],
                  on=['subject', 'model', 'topic', 'ground_truth_num', 'researched_num', 'istest_num', 'searchornot'], how='left')

    # Reorder the dataframe for further processing
    df = df.loc[df['presented'] == 'post']

    # Create numeric subject IDs
    # subject_ids = {subj: i for i, subj in enumerate(df['subject'].unique())}
    # df['subject_id'] = df['subject'].map(subject_ids)
    df['model_subject'] = df['model'] + '_' + df['subject'].astype(str)
    subject_ids = {subj: i for i, subj in enumerate(df['model_subject'].unique())}
    df['subject_id'] = df['model_subject'].map(subject_ids)

    # Define predictor variables
    main_vars = ['ground_truth', 'researched', 'istest']

    # Only include variables with variance in the design matrix
    selected_vars = []
    for var in main_vars:
        var_data = df[f"{var}_num"]
        if var_data.var() > 0:
            selected_vars.append(var)
        else:
            print(f"WARNING: {var} has no variance in the subset - excluding from model")

    # Start with main effects
    X_dict = {'Intercept': np.ones(len(df))}

    # Add variables with variance
    for var in selected_vars:
        X_dict[var] = df[f"{var}_num"]

    # Add sub_compliance
    X_dict['sub_compliance'] = df['sub_compliance_centered']

    if not exclude_models: 
        if prompting == 0:
            # Define effect coding for model categories
            df['model_gpt4o'] = np.where(df['model'] == 'GPT4o', 0.5, np.where(df['model'] == 'claude', -0.25, -0.25))
            df['model_claude'] = np.where(df['model'] == 'claude', 0.5, np.where(df['model'] == 'GPT4o', -0.25, -0.25))
            df['model_mistral'] = np.where(df['model'] == 'mistral', 0.5, np.where(df['model'] == 'GPT4o', -0.25, -0.25))

            # Add effect-coded model columns
            X_dict['model_gpt4o'] = df['model_gpt4o']
            X_dict['model_claude'] = df['model_claude']
            X_dict['model_mistral'] = df['model_mistral']
            
        elif prompting == 1:
            # Define effect coding for model categories
            df['model_sycophancy'] = np.where(df['model'] == 'GPT4o_sycophancy_both', 0.5, -0.5)
            df['model_persuasion'] = np.where(df['model'] == 'GPT4o_persuasion', 0.5, -0.5)
            
            # Add effect-coded model columns
            X_dict['model_sycophancy'] = df['model_sycophancy']
            X_dict['model_persuasion'] = df['model_persuasion']

    X = pd.DataFrame(X_dict)

    # Define a function to add interaction terms
    def add_interactions(X, order, vars_to_interact, model_vars=None):
        print(f"\nAdding {order}-way interactions...")

        # Get all combinations of vars_to_interact of the given order
        for var_combo in combinations(vars_to_interact, order):
            # Create the interaction name
            interaction_name = '_'.join(var_combo)

            # Create the base interaction term
            X[interaction_name] = 1.0
            for var in var_combo:
                X[interaction_name] *= X[var]

            # Add interactions with model variables if provided
            if model_vars:
                for model_var in model_vars:
                    model_interaction_name = f"{interaction_name}_{model_var}"
                    X[model_interaction_name] = X[interaction_name] * X[model_var]

        return X

    # Only proceed with interactions if we have variables with variance
    if not exclude_models:
        if prompting == 0:
            model_vars = ['model_gpt4o', 'model_claude', 'model_mistral']
        elif prompting == 1:
            model_vars = ['model_sycophancy', 'model_persuasion']
    else:
        model_vars = []
        
    if len(selected_vars) >= 2:
        # Interactions incrementally by order
        max_order = min(interaction_level, len(selected_vars))
        for order in range(2, max_order + 1):
            X = add_interactions(X, order, selected_vars, model_vars)
    else:
        print("WARNING: Not enough variables with variance to create interactions.")

    # Check for numerical issues
    print("\nChecking for numerical issues in design matrix...")
    var_threshold = 1e-10
    low_var_cols = []

    for col in X.columns:
        var_val = X[col].var()
        if var_val < var_threshold and col != 'Intercept':
            low_var_cols.append((col, var_val))

    if low_var_cols:
        print("WARNING: The following columns have very low variance:")
        for col, var_val in low_var_cols:
            print(f"  {col}: {var_val:.10e}")
        # Remove columns with no variance - keep Intercept
        cols_to_keep = [col for col in X.columns if X[col].var() >= var_threshold or col == 'Intercept']
        if cols_to_keep:
            X = X[cols_to_keep]
            print(f"Removed {len(low_var_cols)} columns with no variance.")
        else:
            print("ERROR: No columns with variance left after filtering!")

    # Check for multicollinearity
    try:
        from numpy.linalg import svd
        U, s, Vh = svd(X.values, full_matrices=False)
        condition_number = s[0] / s[-1]
        print(f"Condition number of design matrix: {condition_number:.4e}")

        if condition_number > 1e10:
            print("WARNING: Design matrix is poorly conditioned. Consider simplifying the model.")
    except Exception as e:
        print(f"Could not compute condition number: {e}")

    # Extract response variable
    y = df['sign_flip'].values

    # Print summary statistics
    print("\nResponse variable stats:")
    print(f"  Range: {np.min(y)} to {np.max(y)}")
    print(f"  Mean: {np.mean(y):.4f}, Std: {np.std(y):.4f}")
    print(f"  Unique values: {sorted(np.unique(y))}")

    # Subject IDs for random effects
    subject_ids_array = df['subject_id'].values
    num_subjects = len(np.unique(subject_ids_array))
    print(f"\nNumber of subjects: {num_subjects}")

    # Check data per subject
    subject_counts = df.groupby('subject_id').size()
    print(f"Observations per subject: min={subject_counts.min()}, max={subject_counts.max()}, mean={subject_counts.mean():.1f}")

    # Convert to JAX arrays with proper data types
    X_array = jnp.array(X.values, dtype=jnp.float32)

    # If your model is expecting integer outcomes
    y_array = jnp.array(y.astype(np.int32))

    # Subject IDs must be integers
    subject_ids_array = jnp.array(subject_ids_array, dtype=jnp.int32)

    # For ordinal models (if applicable)
    unique_y = np.sort(np.unique(y_array))
    num_cutpoints = len(unique_y) - 1

    print(f"\nFinal design matrix shape: {X_array.shape}")
    print(f"Number of parameters: {X_array.shape[1]}")
    print(f"Response variable type: {y_array.dtype}")
    print(f"Subject IDs type: {subject_ids_array.dtype}")

    return X_array, y_array, subject_ids_array, num_subjects, num_cutpoints, X.columns.tolist(), unique_y


#MCMC fitting function
def run_inference(model, X, y, subject_ids, num_subjects, num_cutpoints, rng_key, 
                 num_warmup=500, num_samples=500, num_chains=4, 
                 target_accept_prob=0.85, max_tree_depth=8, 
                 progress_bar=True, chain_method='parallel'):
    
    rng_keys = random.split(rng_key, num_chains)
    
    # Configure NUTS for models with many parameters
    kernel = NUTS(
        model,
        target_accept_prob=target_accept_prob,
        max_tree_depth=max_tree_depth,
        init_strategy=init_to_feasible
    )
    
    # Create MCMC object
    mcmc = MCMC(
        kernel, 
        num_warmup=num_warmup, 
        num_samples=num_samples, 
        num_chains=num_chains, 
        progress_bar=progress_bar,
        chain_method=chain_method
    )
    
    mcmc.run(rng_keys, X, y, subject_ids, num_subjects, num_cutpoints)
    
    return mcmc


#parameter summary function
def summarize_results(mcmc, feature_names, focus_on_interactions=True):
    """
    Summarize the model results with special focus on interactions if requested.
    """
    # Get samples
    samples = mcmc.get_samples()
    
    if 'beta' in samples:
        beta_samples = samples['beta']
        num_features = beta_samples.shape[1]
        
        # Adjust feature names if needed
        if num_features < len(feature_names):
            feature_names = feature_names[:num_features]
        
        # Calculate summaries
        means = np.mean(beta_samples, axis=0)
        hdi_low = np.percentile(beta_samples, 2.5, axis=0)
        hdi_high = np.percentile(beta_samples, 97.5, axis=0)
    
        # Create summary dataframe
        summary = pd.DataFrame({
            'Parameter': feature_names,
            'Mean': means,
            'HDI_2.5%': hdi_low,
            'HDI_97.5%': hdi_high,
        })
        
        # If focusing on interactions, filter and sort
        if focus_on_interactions:
            # Identify interaction terms
            interaction_mask = summary['Parameter'].str.contains('_')
            
            # Split into main effects and interactions
            main_effects = summary[~interaction_mask].copy()
            interactions = summary[interaction_mask].copy()
            
            # Recombine with main effects at the top
            summary = pd.concat([main_effects, interactions])
        
        # Add random effects variance
        if 'intercept_subject' in samples:
            intercept_subject_mean = np.mean(samples['intercept_subject'])
            intercept_subject_hdi_low = np.percentile(samples['intercept_subject'], 2.5)
            intercept_subject_hdi_high = np.percentile(samples['intercept_subject'], 97.5)
            
            summary = pd.concat([
                summary,
                pd.DataFrame({
                    'Parameter': ['intercept_subject'],
                    'Mean': [intercept_subject_mean],
                    'HDI_2.5%': [intercept_subject_hdi_low],
                    'HDI_97.5%': [intercept_subject_hdi_high],
                })
            ])
        return summary
    

# plot function for all parameters with 95% HDI (forest plot)
def plot_forestplot(summary_df, figsize=(10, 16), rope_interval=rope_interval, outcome=f"{dataset}_full_GLM"):
    # Sort by mean
    summary_df = summary_df.sort_values('Mean', ascending=True)

    #exclude Intercept
    summary_df = summary_df[summary_df['Parameter'] != 'Intercept']

    # # Sort by parameter names alphabetically
    # summary_df = summary_df.sort_values('Parameter', ascending=True)

    # Calculate xerr as the difference from the mean to the confidence bounds
    xerr = np.abs(summary_df[['HDI_2.5%', 'HDI_97.5%']].T.values - summary_df['Mean'].values)

    # Define the figure
    plt.figure(figsize=figsize)
    plt.errorbar(
        summary_df['Mean'], 
        range(summary_df.shape[0]), 
        xerr=xerr,
        fmt='o', 
        color='black', 
        markersize=4,
    )
    
    # Add a vertical line at 0
    plt.axvline(0, color='gray', linestyle='--', linewidth=0.5)

    # Add the shaded zone for the ROPE
    plt.axvspan(rope_interval[0], rope_interval[1], color='gray', alpha=0.3, label='ROPE')
    
    # Add labels
    plt.yticks(range(summary_df.shape[0]), summary_df['Parameter'])
    plt.xlabel('Effect Size')
    #save
    plt.savefig(f'../plots/forest_plot_full_GLM_{outcome}.png', dpi=300, bbox_inches='tight')
    plt.show()


def calculate_overlap(rope_interval, hdi_lower, hdi_upper):
    rope_lower, rope_upper = rope_interval

    # Calculate the intersection range
    overlap_lower = max(rope_lower, hdi_lower)
    overlap_upper = min(rope_upper, hdi_upper)

    # Ensure that there is an overlap
    if overlap_lower > overlap_upper:
        return 0.0  # No overlap

    # Length of the overlap
    overlap_length = overlap_upper - overlap_lower

    # Normalize overlap by the length of the HPDI (or ROPE length if you prefer)
    hdi_length = hdi_upper - hdi_lower
    normalized_overlap = overlap_length / hdi_length

    return normalized_overlap

def compute_waic(log_likelihood_samples):
    """
    Manually compute WAIC from log-likelihood samples.
    Computed on CPU to avoid GPU OOM.
    log_likelihood_samples: shape (n_samples, n_obs)
    """
    from scipy.special import logsumexp as scipy_logsumexp
    
    # Ensure numpy array on CPU
    ll = np.array(log_likelihood_samples)
    n_samples, n_obs = ll.shape
    
    # lppd: log pointwise predictive density
    lppd_per_obs = scipy_logsumexp(ll, axis=0) - np.log(n_samples)
    lppd = float(np.sum(lppd_per_obs))
    
    # Effective number of parameters
    p_waic_per_obs = np.var(ll, axis=0)
    p_waic = float(np.sum(p_waic_per_obs))
    
    # ELPD
    elpd_waic = lppd - p_waic
    
    # SE of ELPD
    elpd_per_obs = lppd_per_obs - p_waic_per_obs
    se = float(np.sqrt(n_obs * np.var(elpd_per_obs)))
    
    # WAIC (on deviance scale)
    waic = -2 * elpd_waic
    
    return {
        'waic': waic, 'elpd_waic': elpd_waic, 'p_waic': p_waic,
        'se': se, 'lppd': lppd, 'elpd_per_obs': elpd_per_obs
    }


def compare_waic(model_dict):
    """
    Compare models via WAIC, outputting a DataFrame in ArviZ-like format.
    model_dict: dict of {model_name: mcmc_object}
    """
    results = {}
    for name, mcmc in model_dict.items():
        ll = np.array(mcmc.get_samples()['log_likelihood'])
        results[name] = compute_waic(ll)
    
    # Find best model
    best_elpd = max(r['elpd_waic'] for r in results.values())
    
    # Compute dse (SE of difference) relative to best model
    best_name = [k for k, v in results.items() if v['elpd_waic'] == best_elpd][0]
    best_elpd_per_obs = results[best_name]['elpd_per_obs']
    
    rows = []
    for name, r in results.items():
        elpd_diff = best_elpd - r['elpd_waic']
        
        if name == best_name:
            dse = 0.0
        else:
            diff_per_obs = best_elpd_per_obs - r['elpd_per_obs']
            n_obs = len(diff_per_obs)
            dse = float(np.sqrt(n_obs * np.var(diff_per_obs)))
        
        # Compute weights
        rows.append({
            'name': name,
            'elpd_waic': r['elpd_waic'],
            'p_waic': r['p_waic'],
            'elpd_diff': elpd_diff,
            'se': r['se'],
            'dse': dse,
        })
    
    df = pd.DataFrame(rows)
    df = df.sort_values('elpd_diff').reset_index(drop=True)
    df['rank'] = range(len(df))
    
    # Compute Akaike-like weights
    elpd_values = df['elpd_waic'].values
    weights = np.exp(elpd_values - elpd_values.max())
    weights = weights / weights.sum()
    df['weight'] = weights
    
    df['warning'] = False
    df['scale'] = 'log'
    
    # Set index to model name
    df = df.set_index('name')
    df = df[['rank', 'elpd_waic', 'p_waic', 'elpd_diff', 'weight', 'se', 'dse', 'warning', 'scale']]
    
    return df

#### Model definitions and fitting

In [ ]:
def ordinal_model(X=None, y=None, subject_ids=None, num_subjects=None, num_cutpoints=None):
    # Non-centered parameterization for fixed effects
    beta_scale = numpyro.sample('beta_scale', dist.HalfNormal(0.1))
    beta_raw = numpyro.sample('beta_raw', dist.Normal(0, 0.1), sample_shape=(X.shape[1],))
    beta = numpyro.deterministic('beta', beta_raw * beta_scale)
    
    # Non-centered parameterization for subject-level effects
    sigma_subject_raw = numpyro.sample('sigma_subject_raw', dist.HalfNormal(0.01))
    sigma_subject = numpyro.deterministic('sigma_subject', sigma_subject_raw * 2.0)
    
    with numpyro.plate('subjects', num_subjects):
        subject_offset_raw = numpyro.sample('subject_offset_raw', dist.Normal(0, 0.01))
        subject_intercept = subject_offset_raw * sigma_subject
    
    # Dirichlet prior for cutpoint spacings
    # Use concentration parameters all equal to alpha for a symmetric prior
    alpha = 1.0  # 1.0, Can be adjusted based on prior knowledge
    concentration = jnp.ones(num_cutpoints + 1) * alpha
    
    # Define anchor point (usually 0 for OrderedLogistic)
    anchor_point = 0.0
    
    # Use TransformReparam for cutpoints
    with reparam(config={"cutpoints": TransformReparam()}):
        cutpoints = numpyro.sample(
            "cutpoints",
            dist.TransformedDistribution(
                dist.Dirichlet(concentration),
                transforms.SimplexToOrderedTransform(anchor_point),
            ),
        )

    # Linear predictor
    eta = jnp.dot(X, beta) + subject_intercept[subject_ids]
    
    # Ordinal likelihood
    numpyro.sample('y', dist.OrderedLogistic(eta, cutpoints), obs=y)
    if y is not None:
        numpyro.deterministic('log_likelihood', dist.OrderedLogistic(eta, cutpoints).log_prob(y))

#Binomial model to fit extremism data
def binomial_model(X=None, y=None, subject_ids=None, num_subjects=None, num_cutpoints=None, n_trials=1):
    # Non-centered parameterization for fixed effects
    beta_scale = numpyro.sample('beta_scale', dist.HalfNormal(.1))
    beta_raw = numpyro.sample('beta_raw', dist.Normal(0, .1), sample_shape=(X.shape[1],))
    beta = numpyro.deterministic('beta', beta_raw * beta_scale)
    
    # Non-centered parameterization for subject-level effects
    sigma_subject_raw = numpyro.sample('sigma_subject_raw', dist.HalfNormal(.1))
    sigma_subject = numpyro.deterministic('sigma_subject', sigma_subject_raw * 2.0)
    
    with numpyro.plate('subjects', num_subjects):
        subject_offset_raw = numpyro.sample('subject_offset_raw', dist.Normal(0, .1))
        subject_intercept = subject_offset_raw * sigma_subject

    # Linear predictor
    eta = jnp.dot(X, beta) + subject_intercept[subject_ids]
    
    # Logistic (sigmoid) transformation
    p = jax.nn.sigmoid(eta)
    
    # Binomial likelihood with varying n_trials
    numpyro.sample('y', dist.Binomial(n_trials, p), obs=y)
    if y is not None:
        numpyro.deterministic('log_likelihood', dist.Binomial(n_trials, p).log_prob(y))

#### Prior predictive checks

In [ ]:
# Mock data to use in prior predictive check
num_observations = 15000  # Adjust based on your needs
X_mock = jax.random.normal(jax.random.PRNGKey(0), (num_observations, 17))  # num_observations with 17 predictors
subject_ids_mock = jax.random.randint(jax.random.PRNGKey(1), (num_observations,), 0, 1200)  # subjects
num_subjects_mock = len(jnp.unique(subject_ids_mock))
num_cutpoints = 6  # 6 cutpoints as we split into 7 categories

# Prior predictive samples
num_samples_priorpc = 1000
predictive = Predictive(ordinal_model, num_samples=num_samples_priorpc)
prior_samples = predictive(jax.random.PRNGKey(2), X=X_mock, subject_ids=subject_ids_mock, num_subjects=num_subjects_mock, num_cutpoints=num_cutpoints)

# Plotting the prior predictive check for y
y_samples = prior_samples['y']
plt.figure(figsize=(10, 6))
plt.hist(y_samples.flatten(), bins=num_cutpoints+1, density=True, alpha=0.75, edgecolor='k')
plt.title('Prior Predictive Check for Ordinal Model')
plt.xlabel('y value')
plt.ylabel('Density')
plt.show()

In [ ]:
# Mock data to use in prior predictive check
num_observations = 15000  # Adjust based on the desired number of observations
num_predictors = 17  # Number of predictors
num_subjects = 1200  # Number of subjects

X_mock = jax.random.normal(jax.random.PRNGKey(0), (num_observations, num_predictors))  # Mock input data
subject_ids_mock = jax.random.randint(jax.random.PRNGKey(1), (num_observations,), 0, num_subjects)
n_trials_mock = 10 

# Prior predictive samples
num_samples_priorpc = 1000
predictive = Predictive(binomial_model, num_samples=num_samples_priorpc)
prior_samples = predictive(jax.random.PRNGKey(2), X=X_mock, subject_ids=subject_ids_mock, num_subjects=num_subjects, n_trials=n_trials_mock)

# Plotting the prior predictive check for y
y_samples = prior_samples['y']
plt.figure(figsize=(10, 6))
plt.hist(y_samples.flatten(), bins=min(n_trials_mock, 50), density=True, alpha=0.75, edgecolor='k')
plt.title('Prior Predictive Check for Binomial Model')
plt.xlabel('y value')
plt.ylabel('Density')
plt.show()

### Run all models

In [ ]:
#run across datasets
for dataset in ["misinfo", "trust", "private", "extreme"]:

    if dataset == "misinfo":    
        data = pd.read_csv('../data/Ztable_misinfo_combined.csv')
    elif dataset == "trust":
        data = pd.read_csv('../data/Ztable_trust_combined.csv')
    elif dataset == "private":
        data = pd.read_csv('../data/Ztable_private_combined.csv')
    elif dataset == "extreme":
        data = pd.read_csv('../data/Ztable_extreme_combined.csv')

    print(len(data))
    print(data["model"].unique())

    #run across prompting
    for prompting in [0, 1]:
        #run across exclude_models
        for exclude_models in [0, 1]:

            print("Working on dataset: ", dataset)
            print("Excluding models: ", exclude_models)
            print("Prompting: ", prompting)
    
            # Full GLM
            if dataset == "misinfo" or dataset == "trust" or dataset == "private":
                X, y, subject_ids, num_subjects, num_cutpoints, feature_names, unique_y = prepare_data(
                    data, subset_fraction=0.10, use_subset=False, interaction_level=6, dataset=dataset, exclude_models=exclude_models, prompting=prompting)
            elif dataset == "extreme":
                X, y, subject_ids, num_subjects, num_cutpoints, feature_names, unique_y = prepare_data_extremism(
                    data, subset_fraction=0.10, use_subset=False, interaction_level=6, exclude_models=exclude_models, prompting=prompting)
    

            #plot heatmap for X
            plt.figure(figsize=(8, 6))
            sns.heatmap(X, cmap='viridis', cbar=False)
            plt.xlabel("Predictors")
            plt.ylabel("Observations")
            plt.title("Design Matrix Heatmap")
            plt.show()

            # Calculate VIF for each predictor
            def calculate_vif(X):
                vif_data = pd.DataFrame()
                vif_data["feature"] = feature_names
                vif_data["VIF"] = [variance_inflation_factor(X, i) for i in range(X.shape[1])]
                return vif_data

            # Print VIF scores
            vif_scores = calculate_vif(X)
            print(vif_scores)



            # Run inference 
            if dataset == "misinfo" or dataset == "trust" or dataset == "private":
                mcmc_full = run_inference(
                    model=ordinal_model, 
                    X=X, 
                    y=y, 
                    subject_ids=subject_ids, 
                    num_subjects=num_subjects, 
                    num_cutpoints=num_cutpoints, 
                    rng_key=rng_key,
                    num_warmup=num_warmup, 
                    num_samples=num_samples, 
                    num_chains=num_chains
                )
            elif dataset == "extreme":
                mcmc_full = run_inference(
                        model=binomial_model, 
                        X=X, 
                        y=y, 
                        subject_ids=subject_ids, 
                        num_subjects=num_subjects, 
                        num_cutpoints=None, 
                        rng_key=rng_key,
                        num_warmup=num_warmup, 
                        num_samples=num_samples, 
                        num_chains=num_chains
                    )

            az_data_full = az.from_numpyro(mcmc_full)

            # Get summary focused on interactions
            summary_full_df = summarize_results(mcmc_full, feature_names, focus_on_interactions=True)

            #show summary_df ordered by index
            summary_full_df = summary_full_df.sort_index()

            # Add the overlap to the summary DataFrame
            summary_full_df['rope_overlap'] = summary_full_df.apply(
                lambda row: calculate_overlap(
                    rope_interval,
                    row['HDI_2.5%'],
                    row['HDI_97.5%']
                ),
                axis=1
            )

            print(summary_full_df)

            mcmc_full.print_summary(exclude_deterministic=False)

            # Save the summary DataFrame to a CSV file
            if exclude_models == 0:
                summary_full_df.to_csv(f'../parameter_estimates/summary_full_GLM_{dataset}_prompting_{prompting}.csv', index=False)
                plot_forestplot(summary_full_df, figsize=(10, 16), rope_interval=rope_interval, outcome=f"{dataset}_prompting_{prompting}_full_GLM")

            else:
                summary_full_df.to_csv(f'../parameter_estimates/summary_full_GLM_{dataset}_prompting_{prompting}_no_model.csv', index=False)
                plot_forestplot(summary_full_df, figsize=(6, 8), rope_interval=rope_interval, outcome=f"{dataset}_prompting_{prompting}_full_GLM_no_model")
        
            #plot traceplot for some parameters
            if dataset == "private":
                az.plot_trace(az.from_numpyro(mcmc_full), var_names=["cutpoints"])
            elif dataset == "misinfo" or dataset == "trust":
                az.plot_trace(az.from_numpyro(mcmc_full), var_names=["sigma_subject", "beta_scale", "cutpoints"])
            else:
                az.plot_trace(az.from_numpyro(mcmc_full), var_names=["sigma_subject", "beta_scale"])
            
            if exclude_models == 0:
               mcmc_full_models = mcmc_full
            else:
                mcmc_full_no_models = mcmc_full

            
        #control GLM
        #make dataset
        if dataset == "misinfo" or dataset == "trust" or dataset == "private":
            X, y, subject_ids, num_subjects, num_cutpoints, feature_names, unique_y = prepare_data(
                data, subset_fraction=0.10, use_subset=False, interaction_level=6, dataset=dataset, exclude_models=exclude_models, prompting=prompting)
        elif dataset == "extreme":
            X, y, subject_ids, num_subjects, num_cutpoints, feature_names, unique_y = prepare_data_extremism(
                data, subset_fraction=0.10, use_subset=False, interaction_level=6, exclude_models=exclude_models, prompting=prompting)

        # Specify the variables to exclude, everyting that contains 'iscontrol' and model variables
        exclude_vars = ['istest',
                        'model_gpt4o',
                        'model_claude',
                        'model_mistral',
                        'ground_truth_researched_model_gpt4o',
                        'ground_truth_researched_model_claude',
                        'ground_truth_researched_model_mistral',
                        'ground_truth_istest',
                        'ground_truth_istest_model_gpt4o',
                        'ground_truth_istest_model_claude',
                        'ground_truth_istest_model_mistral',
                        'researched_istest',
                        'researched_istest_model_gpt4o',
                        'researched_istest_model_claude',
                        'researched_istest_model_mistral',
                        'ground_truth_researched_istest',
                        'ground_truth_researched_istest_model_gpt4o',
                        'ground_truth_researched_istest_model_claude',
                        'ground_truth_researched_istest_model_mistral',
                        'time_istest',
                        'time_model_gpt4o',
                        'time_model_claude',
                        'time_model_mistral',
                        'time_ground_truth_model_gpt4o',
                        'time_ground_truth_model_claude',
                        'time_ground_truth_model_mistral',
                        'time_researched_model_gpt4o',
                        'time_researched_model_claude',
                        'time_researched_model_mistral',
                        'time_istest_model_gpt4o',
                        'time_istest_model_claude',
                        'time_istest_model_mistral',
                        'time_ground_truth_researched_model_gpt4o',
                        'time_ground_truth_researched_model_claude',
                        'time_ground_truth_researched_model_mistral',
                        'time_ground_truth_istest',
                        'time_ground_truth_istest_model_gpt4o',
                        'time_ground_truth_istest_model_claude',
                        'time_ground_truth_istest_model_mistral',
                        'time_researched_istest',
                        'time_researched_istest_model_gpt4o',
                        'time_researched_istest_model_claude',
                        'time_researched_istest_model_mistral',
                        'time_ground_truth_researched_istest',
                        'time_ground_truth_researched_istest_model_gpt4o',
                        'time_ground_truth_researched_istest_model_claude',
                        'time_ground_truth_researched_istest_model_mistral',
                        'model_sycophancy', 'model_persuasion', 
                        'ground_truth_researched_model_sycophancy', 
                        'ground_truth_researched_model_persuasion', 
                        'ground_truth_istest_model_sycophancy', 
                        'ground_truth_istest_model_persuasion',
                        'researched_istest_model_sycophancy', 
                        'researched_istest_model_persuasion', 
                        'ground_truth_researched_istest_model_sycophancy', 
                        'ground_truth_researched_istest_model_persuasion', 
                        'time_model_sycophancy', 'time_model_persuasion', 
                        'time_ground_truth_model_sycophancy', 
                        'time_ground_truth_model_persuasion', 
                        'time_researched_model_sycophancy', 
                        'time_researched_model_persuasion', 
                        'time_istest_model_sycophancy', 
                        'time_istest_model_persuasion', 
                        'time_ground_truth_researched_model_sycophancy', 
                        'time_ground_truth_researched_model_persuasion', 
                        'time_ground_truth_istest_model_sycophancy', 
                        'time_ground_truth_istest_model_persuasion', 
                        'time_researched_istest_model_sycophancy', 
                        'time_researched_istest_model_persuasion', 
                        'time_ground_truth_researched_istest_model_sycophancy', 
                        'time_ground_truth_researched_istest_model_persuasion']


        # Ensure the exclude_vars list is in the same format as feature_names
        exclude_vars = [var for var in exclude_vars if var in feature_names]

        # Create a boolean mask for the columns to keep
        columns_to_keep = [var not in exclude_vars for var in feature_names]

        # Filter X_array to exclude the specified columns
        X_filtered = X[:, columns_to_keep]

        # Filter feature_names to exclude the specified variables
        feature_names_filtered = [var for var in feature_names if var not in exclude_vars]

        # If needed, verify the shapes and contents
        print(f"Original X shape: {X.shape}, Filtered X shape: {X_filtered.shape}")
        print(f"Original feature names: {feature_names}")
        print(f"Filtered feature names: {feature_names_filtered}")

        #plot heatmap for X
        plt.figure(figsize=(12, 8))
        sns.heatmap(X_filtered, cmap='viridis', cbar=False)
        plt.xlabel("Predictors")
        plt.ylabel("Observations")
        plt.title("Design Matrix Heatmap")
        plt.show()

        # Run inference 
        if dataset == "misinfo" or dataset == "trust" or dataset == "private":
            mcmc_control = run_inference(
                model=ordinal_model, 
                X=X_filtered, 
                y=y, 
                subject_ids=subject_ids, 
                num_subjects=num_subjects, 
                num_cutpoints=num_cutpoints, 
                rng_key=rng_key,
                num_warmup=num_warmup, 
                num_samples=num_samples, 
                num_chains=num_chains
            )
        elif dataset == "extreme":
            mcmc_control = run_inference(
                    model=binomial_model, 
                    X=X_filtered, 
                    y=y, 
                    subject_ids=subject_ids, 
                    num_subjects=num_subjects, 
                    num_cutpoints=None, 
                    rng_key=rng_key,
                    num_warmup=num_warmup, 
                    num_samples=num_samples, 
                    num_chains=num_chains
                )

        # Get summary focused on interactions
        summary_control_df = summarize_results(mcmc_control, feature_names_filtered, focus_on_interactions=True)
        #show summary_df ordered by index
        summary_control_df = summary_control_df.sort_index()

        # Add the overlap to the summary DataFrame
        summary_control_df['rope_overlap'] = summary_control_df.apply(
            lambda row: calculate_overlap(
                rope_interval,
                row['HDI_2.5%'],
                row['HDI_97.5%']
            ),
            axis=1
        )

        print(summary_control_df)
        mcmc_control.print_summary(exclude_deterministic=False)

        # Save the summary DataFrame to a CSV file
        summary_control_df.to_csv(f'../parameter_estimates/summary_control_GLM_{dataset}_prompting_{prompting}.csv', index=False)

        plot_forestplot(summary_control_df, figsize=(6, 8), rope_interval=rope_interval, outcome=f"{dataset}_prompting_{prompting}_control_GLM")

        #plot traceplot for some parameters
        if dataset == "private":
            az.plot_trace(az.from_numpyro(mcmc_control), var_names=["cutpoints"])
        elif dataset == "misinfo" or dataset == "trust":
            az.plot_trace(az.from_numpyro(mcmc_control), var_names=["sigma_subject", "beta_scale", "cutpoints"])
        else:
            az.plot_trace(az.from_numpyro(mcmc_control), var_names=["sigma_subject", "beta_scale"])

        comparison = compare_waic({
            "GLM_full_with_model": mcmc_full_models,
            "GLM_full_without_model": mcmc_full_no_models,
            "GLM_control": mcmc_control
        })
        
        print(comparison)
        comparison.to_csv(f'../parameter_estimates/waic_comparison_{dataset}_prompting_{prompting}.csv')

# Analyses on progressive shift - relation to party identification?

In [ ]:
data_beliefs = pd.read_csv('../data/Ztable_private_combined.csv')

print(len(data_beliefs))
print(data_beliefs.head(10))

data_demo = pd.read_csv('../data/Ztable_indiv_combined.csv')

print(len(data_demo))
print(data_demo.head(10))

In [ ]:
# Step 1: Assign subject IDs to demographics
data_demo['subject'] = data_demo.groupby('model').cumcount() + 1

# Step 2: Check the merge keys are unique in demographics
print(f"Unique model-subject combos in demographics: {data_demo.groupby(['model', 'subject']).ngroups}")
print(f"Total rows in demographics: {len(data_demo)}")

# Step 3: Merge
merged = data_beliefs.merge(data_demo, on=['model', 'subject'], how='left', suffixes=('_outcome', '_demo'))

# Step 4: Validate
print(f"\nRows in outcomes before merge: {len(data_beliefs)}")
print(f"Rows after merge: {len(merged)}")
print(f"Any NaN in demographics columns after merge: {merged['age'].isna().sum()}")

# Step 5: Validate with offsets
offsets = {
    'GPT4o': 0,
    'mistral': 1,
    'claude': 2,
    'GPT4o_sycophancy_both': 2.9,
    'GPT4o_persuasion': 3.9
}
merged['expected_compliance'] = merged['model'].map(offsets) + merged['compliance']
match_rate = ((merged['sub_compliance'] - merged['expected_compliance']).abs() < 0.001).mean()
print(f"\nOffset-adjusted compliance match rate: {match_rate:.4%}")

# Step 6: Preview
print("\nFirst few rows of key columns:")
print(merged[['model', 'subject', 'age', 'gender', 'vote', 'education', 
              'iscontrol', 'response', 'presented', 'ground_truth']].head(10))

In [ ]:
# Check what columns they share
print("Beliefs columns:", data_beliefs.columns.tolist())
print("\nDemo columns:", data_demo.columns.tolist())
print("\nShared columns:", set(data_beliefs.columns) & set(data_demo.columns))

# Check searchornot in both
print("\nSearchornot in beliefs:")
print(data_beliefs['searchornot'].value_counts())
print("\nSearchornot in demo:")
print(data_demo['searchornot'].value_counts())

# Assign subject IDs
data_demo['subject'] = data_demo.groupby('model').cumcount() + 1

# Merge
merged = data_beliefs.merge(data_demo, on=['model', 'subject'], how='left', suffixes=('_outcome', '_demo'))

# Triple validation
# 1. Offset-adjusted compliance
offsets = {
    'GPT4o': 0,
    'mistral': 1,
    'claude': 2,
    'GPT4o_sycophancy_both': 2.9,
    'GPT4o_persuasion': 3.9
}
merged['expected_compliance'] = merged['model'].map(offsets) + merged['compliance']
compliance_match = ((merged['sub_compliance'] - merged['expected_compliance']).abs() < 0.001).mean()

# 2. Searchornot match
searchornot_match = (merged['searchornot_outcome'] == merged['searchornot_demo']).mean()

# 3. Check a few random subjects manually
print(f"\nCompliance match rate: {compliance_match:.4%}")
print(f"Searchornot match rate: {searchornot_match:.4%}")

# 4. Spot check: for each model, show first 5 subjects side by side
for m in data_demo['model'].unique():
    demo_sub = data_demo[data_demo['model'] == m].head(5)[['model', 'subject', 'compliance', 'searchornot']]
    beliefs_sub = data_beliefs[data_beliefs['model'] == m].drop_duplicates('subject').head(5)[['model', 'subject', 'sub_compliance', 'searchornot']]
    print(f"\n--- {m} ---")
    print("Demo:")
    print(demo_sub.to_string())
    print("Beliefs:")
    print(beliefs_sub.to_string())

In [ ]:
merged

In [ ]:
# Step 1: Merge demographics into beliefs data
data_demo['subject'] = data_demo.groupby('model').cumcount() + 1
data_beliefs = data_beliefs.merge(data_demo[['model', 'subject', 'vote']], on=['model', 'subject'], how='left')

# Step 2: Create binary vote variable
progressive = ['labour', 'green', 'liberal', 'SNP', 'sinn_fein', 'plaid_cymru']
conservative = ['conservative', 'reform_UK', 'unionist']
ambiguous = ['prefer_not_to_say', 'dont_know', 'other', 'missing']

data_beliefs['vote_binary'] = np.where(
    data_beliefs['vote'].isin(progressive), 0.5,
    np.where(data_beliefs['vote'].isin(conservative), -0.5, np.nan)
)

# Check distribution
print("Vote binary distribution (at observation level):")
print(data_beliefs['vote_binary'].value_counts(dropna=False))
print(f"\nObservations dropped: {data_beliefs['vote_binary'].isna().sum()} ({data_beliefs['vote_binary'].isna().mean():.1%})")

# Step 3: Verify merge worked
print(f"\nTotal rows: {len(data_beliefs)}")
print(f"NaN in vote: {data_beliefs['vote'].isna().sum()}")

In [ ]:
# At subject level (not observation level)
subject_level = data_beliefs.drop_duplicates(['model', 'subject'])

print("=== Subject-level vote distribution ===")
print(f"\nTotal subjects: {len(subject_level)}")

# Detailed
print("\nDetailed vote:")
print(subject_level['vote'].value_counts())

# Binary
print("\nBinary vote:")
n_progressive = (subject_level['vote_binary'] == 0.5).sum()
n_conservative = (subject_level['vote_binary'] == -0.5).sum()
n_ambiguous = subject_level['vote_binary'].isna().sum()
n_total = len(subject_level)

print(f"Progressive: {n_progressive} ({n_progressive/n_total:.1%})")
print(f"Conservative: {n_conservative} ({n_conservative/n_total:.1%})")
print(f"Ambiguous/excluded: {n_ambiguous} ({n_ambiguous/n_total:.1%})")

# Among those with clear ideology
n_clear = n_progressive + n_conservative
print(f"\nAmong classified voters (n={n_clear}):")
print(f"Progressive: {n_progressive} ({n_progressive/n_clear:.1%})")
print(f"Conservative: {n_conservative} ({n_conservative/n_clear:.1%})")

In [ ]:
def prepare_data_progressive(df, use_subset=False, subset_fraction=0.01,
                              interaction_level=5, exclude_models=True):
    """
    Prepare private beliefs data with vote_binary added as a main variable,
    fully interacted with existing predictors including time.
    Only for the base models (no sycophancy/persuasion prompting).
    """
    print("\nDistribution in raw data:")
    print(f"ground_truth values: {df['ground_truth'].value_counts().to_dict()}")
    print(f"researched values: {df['researched'].value_counts().to_dict()}")
    print(f"iscontrol values: {df['iscontrol'].value_counts().to_dict()}")

    # Keep only base models
    df = df[~df['model'].isin(['GPT4o_sycophancy_both', 'GPT4o_persuasion'])].copy()

    # Drop ambiguous vote categories
    n_before = len(df)
    df = df[df['vote_binary'].notna()].copy()
    print(f"\nObservations dropped (ambiguous vote): {n_before - len(df)} ({(n_before - len(df))/n_before:.1%})")
    print(f"Observations remaining: {len(df)}")
    print(f"  Progressive (+0.5): {(df['vote_binary'] == 0.5).sum()}")
    print(f"  Conservative (-0.5): {(df['vote_binary'] == -0.5).sum()}")

    # Create time variable
    df['time'] = (df['presented'] == 'post').astype(float) - 0.5

    # Create numeric versions
    df['ground_truth_num'] = (df['ground_truth'] == True).astype(float) - 0.5
    df['researched_num'] = (df['researched'] == 'yes').astype(float) - 0.5
    df['istest_num'] = (df['iscontrol'] == 0).astype(float) - 0.5
    df['vote_num'] = df['vote_binary']

    if use_subset:
        true_subjects = df[df['ground_truth'] == True]['subject'].unique()
        false_subjects = df[df['ground_truth'] == False]['subject'].unique()
        true_sample_size = min(max(int(len(true_subjects) * subset_fraction), 3), len(true_subjects))
        false_sample_size = min(max(int(len(false_subjects) * subset_fraction), 3), len(false_subjects))
        sampled_subjects = np.concatenate([
            np.random.choice(true_subjects, true_sample_size, replace=False),
            np.random.choice(false_subjects, false_sample_size, replace=False)
        ])
        df = df[df['subject'].isin(sampled_subjects)]
        print(f"\nUsing subset: {len(df)} observations from {len(sampled_subjects)} subjects")

    # Center compliance
    df['sub_compliance_centered'] = (df['sub_compliance'] - df['sub_compliance'].mean()) / df['sub_compliance'].std()

    # Create numeric subject IDs
    # subject_ids = {subj: i for i, subj in enumerate(df['subject'].unique())}
    # df['subject_id'] = df['subject'].map(subject_ids)
    df['model_subject'] = df['model'] + '_' + df['subject'].astype(str)
    subject_ids = {subj: i for i, subj in enumerate(df['model_subject'].unique())}
    df['subject_id'] = df['model_subject'].map(subject_ids)


    # Define predictor variables — now including vote
    main_vars = ['ground_truth', 'researched', 'istest', 'vote']

    # Check variance
    selected_vars = []
    for var in main_vars:
        if df[f"{var}_num"].var() > 0:
            selected_vars.append(var)
        else:
            print(f"WARNING: {var} has no variance - excluding")

    # Build design matrix
    X_dict = {'Intercept': np.ones(len(df)), 'time': df['time']}
    for var in selected_vars:
        X_dict[var] = df[f"{var}_num"]
    X_dict['sub_compliance'] = df['sub_compliance_centered']

    # Model variables
    model_vars = []
    if not exclude_models:
        df['model_gpt4o'] = np.where(df['model'] == 'GPT4o', 0.5, np.where(df['model'] == 'claude', -0.25, -0.25))
        df['model_claude'] = np.where(df['model'] == 'claude', 0.5, np.where(df['model'] == 'GPT4o', -0.25, -0.25))
        df['model_mistral'] = np.where(df['model'] == 'mistral', 0.5, np.where(df['model'] == 'GPT4o', -0.25, -0.25))
        model_vars = ['model_gpt4o', 'model_claude', 'model_mistral']
        for mv in model_vars:
            X_dict[mv] = df[mv]

    X = pd.DataFrame(X_dict)

    # Add interaction terms (reusing your existing function)
    def add_interactions(X, order, vars_to_interact, model_vars=None):
        print(f"\nAdding {order}-way interactions...")
        for var_combo in combinations(vars_to_interact, order):
            interaction_name = '_'.join(var_combo)
            X[interaction_name] = 1.0
            for var in var_combo:
                X[interaction_name] *= X[var]
            if model_vars:
                for model_var in model_vars:
                    model_interaction_name = f"{interaction_name}_{model_var}"
                    X[model_interaction_name] = X[interaction_name] * X[model_var]
        return X

    if len(selected_vars) >= 2:
        max_order = min(interaction_level, len(selected_vars))
        for order in range(2, max_order + 1):
            X = add_interactions(X, order, selected_vars, model_vars)

        # Time interactions
        print("\nAdding time interactions...")
        for var in selected_vars:
            X[f"time_{var}"] = X['time'] * X[var]
        if model_vars:
            for model_var in model_vars:
                X[f"time_{model_var}"] = X['time'] * X[model_var]
            for var in selected_vars:
                for model_var in model_vars:
                    X[f"time_{var}_{model_var}"] = X['time'] * X[var] * X[model_var]

        # Higher-order time interactions
        if interaction_level >= 3 and len(selected_vars) >= 2:
            for order in range(2, min(interaction_level, len(selected_vars)) + 1):
                print(f"Adding {order+1}-way time interactions (time x {order} vars)...")
                for var_combo in combinations(selected_vars, order):
                    interaction_name = 'time_' + '_'.join(var_combo)
                    X[interaction_name] = X['time']
                    for var in var_combo:
                        X[interaction_name] *= X[var]
                    if model_vars:
                        for model_var in model_vars:
                            X[f"{interaction_name}_{model_var}"] = X[interaction_name] * X[model_var]

    # Check for low variance columns
    print("\nChecking for numerical issues in design matrix...")
    var_threshold = 1e-10
    low_var_cols = [(col, X[col].var()) for col in X.columns 
                    if X[col].var() < var_threshold and col != 'Intercept']
    if low_var_cols:
        print(f"WARNING: {len(low_var_cols)} columns with very low variance:")
        for col, v in low_var_cols:
            print(f"  {col}: {v:.10e}")
        X = X[[col for col in X.columns if X[col].var() >= var_threshold or col == 'Intercept']]
        print(f"Removed {len(low_var_cols)} columns.")

    # Condition number
    try:
        from numpy.linalg import svd
        _, s, _ = svd(X.values, full_matrices=False)
        print(f"Condition number: {s[0]/s[-1]:.4e}")
    except Exception as e:
        print(f"Could not compute condition number: {e}")

    # Response variable — reverse coded
    y = 6 - (df['response'].values - 1)

    print("\nResponse variable stats:")
    print(f"  Range: {np.min(y)} to {np.max(y)}")
    print(f"  Mean: {np.mean(y):.4f}, Std: {np.std(y):.4f}")
    for time_val, time_name in [(-0.5, "Pre"), (0.5, "Post")]:
        time_responses = y[df['time'] == time_val]
        print(f"  {time_name}: n={len(time_responses)}, mean={np.mean(time_responses):.4f}")

    # Subject IDs
    subject_ids_array = df['subject_id'].values
    num_subjects = len(np.unique(subject_ids_array))
    print(f"\nNumber of subjects: {num_subjects}")
    subject_counts = df.groupby('subject_id').size()
    print(f"Observations per subject: min={subject_counts.min()}, max={subject_counts.max()}, mean={subject_counts.mean():.1f}")

    # Convert to JAX arrays
    X_array = jnp.array(X.values, dtype=jnp.float32)
    y_array = jnp.array(y.astype(np.int32))
    subject_ids_array = jnp.array(subject_ids_array, dtype=jnp.int32)

    unique_y = np.sort(np.unique(y_array))
    num_cutpoints = len(unique_y) - 1

    print(f"\nFinal design matrix shape: {X_array.shape}")
    print(f"Number of parameters: {X_array.shape[1]}")
    print(f"Number of cutpoints: {num_cutpoints}")

    # Print the vote-related parameters for easy reference
    vote_params = [col for col in X.columns if 'vote' in col]
    print(f"\nVote-related parameters ({len(vote_params)}):")
    for p in vote_params:
        print(f"  {p}")

    return X_array, y_array, subject_ids_array, num_subjects, num_cutpoints, X.columns.tolist(), unique_y

In [ ]:
num_samples = 2000
num_warmup = 2000

# ---- MODEL 1: Full model WITH vote ----
print("\n" + "="*60)
print("MODEL 1: Full model WITH vote interactions")
print("="*60)

X_vote, y_vote, subject_ids_vote, num_subjects_vote, num_cutpoints_vote, feature_names_vote, unique_y_vote = \
    prepare_data_progressive(data_beliefs, use_subset=False, interaction_level=5, exclude_models=True)

plt.figure(figsize=(8, 6))
sns.heatmap(X_vote, cmap='viridis', cbar=False)
plt.xlabel("Predictors")
plt.ylabel("Observations")
plt.title("Design Matrix: With Vote")
plt.show()

mcmc_vote = run_inference(
    model=ordinal_model,
    X=X_vote, y=y_vote,
    subject_ids=subject_ids_vote,
    num_subjects=num_subjects_vote,
    num_cutpoints=num_cutpoints_vote,
    rng_key=rng_key,
    num_warmup=num_warmup, num_samples=num_samples, num_chains=num_chains
)

summary_vote_df = summarize_results(mcmc_vote, feature_names_vote, focus_on_interactions=True)
summary_vote_df = summary_vote_df.sort_index()
summary_vote_df['rope_overlap'] = summary_vote_df.apply(
    lambda row: calculate_overlap(rope_interval, row['HDI_2.5%'], row['HDI_97.5%']), axis=1
)
print(summary_vote_df)
mcmc_vote.print_summary(exclude_deterministic=False)

summary_vote_df.to_csv('../parameter_estimates/summary_progressive_with_vote.csv', index=False)
plot_forestplot(summary_vote_df, figsize=(10, 20), rope_interval=rope_interval, 
                outcome="progressive_with_vote")

# Print vote-specific parameters
print("\n=== KEY VOTE PARAMETERS ===")
vote_params = summary_vote_df[summary_vote_df['Parameter'].str.contains('vote')]
print(vote_params.to_string())

az.plot_trace(az.from_numpyro(mcmc_vote), var_names=["cutpoints"])
plt.show()

# ---- MODEL 2: Null model WITHOUT vote (same data subset) ----
# We need to refit on the SAME observations (excluding ambiguous voters)
# but without vote in the model
print("\n" + "="*60)
print("MODEL 2: Null model WITHOUT vote (same data subset)")
print("="*60)

# Use prepare_data on the same filtered data
# First, filter the data the same way prepare_data_progressive does
data_novote = data_beliefs.copy()
data_novote = data_novote[~data_novote['model'].isin(['GPT4o_sycophancy_both', 'GPT4o_persuasion'])]
data_novote = data_novote[data_novote['vote_binary'].notna()]
print(f"Observations in null model (same subset): {len(data_novote)}")

X_novote, y_novote, subject_ids_novote, num_subjects_novote, num_cutpoints_novote, feature_names_novote, unique_y_novote = \
    prepare_data(data_novote, subset_fraction=0.10, use_subset=False, interaction_level=6, 
                 dataset="private", exclude_models=True, prompting=0)

plt.figure(figsize=(8, 6))
sns.heatmap(X_novote, cmap='viridis', cbar=False)
plt.xlabel("Predictors")
plt.ylabel("Observations")
plt.title("Design Matrix: Without Vote")
plt.show()

mcmc_novote = run_inference(
    model=ordinal_model,
    X=X_novote, y=y_novote,
    subject_ids=subject_ids_novote,
    num_subjects=num_subjects_novote,
    num_cutpoints=num_cutpoints_novote,
    rng_key=rng_key,
    num_warmup=num_warmup, num_samples=num_samples, num_chains=num_chains
)

summary_novote_df = summarize_results(mcmc_novote, feature_names_novote, focus_on_interactions=True)
summary_novote_df = summary_novote_df.sort_index()
summary_novote_df['rope_overlap'] = summary_novote_df.apply(
    lambda row: calculate_overlap(rope_interval, row['HDI_2.5%'], row['HDI_97.5%']), axis=1
)
print(summary_novote_df)
mcmc_novote.print_summary(exclude_deterministic=False)

summary_novote_df.to_csv('../parameter_estimates/summary_progressive_without_vote.csv', index=False)
plot_forestplot(summary_novote_df, figsize=(6, 8), rope_interval=rope_interval, 
                outcome="progressive_without_vote")

az.plot_trace(az.from_numpyro(mcmc_novote), var_names=["cutpoints"])
plt.show()





# Get vote-related features to drop (keep only vote and ground_truth_vote)
vote_to_keep = ['vote', 'ground_truth_vote']
vote_to_drop = [f for f in feature_names_vote if 'vote' in f and f not in vote_to_keep]

print(f"Dropping {len(vote_to_drop)} vote interaction terms:")
for f in vote_to_drop:
    print(f"  {f}")

# Filter columns
columns_to_keep = [f not in vote_to_drop for f in feature_names_vote]
X_parsimonious = X_vote[:, columns_to_keep]
feature_names_parsimonious = [f for f in feature_names_vote if f not in vote_to_drop]

print(f"\nFull vote model: {X_vote.shape[1]} parameters")
print(f"Parsimonious model: {X_parsimonious.shape[1]} parameters")
print(f"Features: {feature_names_parsimonious}")

# Fit parsimonious model
mcmc_parsimonious = run_inference(
    model=ordinal_model,
    X=X_parsimonious, y=y_vote,
    subject_ids=subject_ids_vote,
    num_subjects=num_subjects_vote,
    num_cutpoints=num_cutpoints_vote,
    rng_key=rng_key,
    num_warmup=num_warmup, num_samples=num_samples, num_chains=num_chains
)

# Summarize parsimonious model
summary_parsimonious_df = summarize_results(mcmc_parsimonious, feature_names_parsimonious, focus_on_interactions=True)
summary_parsimonious_df = summary_parsimonious_df.sort_index()
summary_parsimonious_df['rope_overlap'] = summary_parsimonious_df.apply(
    lambda row: calculate_overlap(rope_interval, row['HDI_2.5%'], row['HDI_97.5%']), axis=1
)
print(summary_parsimonious_df)
mcmc_parsimonious.print_summary(exclude_deterministic=False)

summary_parsimonious_df.to_csv('../parameter_estimates/summary_progressive_parsimonious.csv', index=False)
plot_forestplot(summary_parsimonious_df, figsize=(8, 12), rope_interval=rope_interval,
                outcome="progressive_parsimonious")



In [ ]:
# ---- 3-WAY MODEL COMPARISON ----
print("\n" + "="*60)
print("MODEL COMPARISON: 3 models (WAIC)")
print("="*60)

comparison_3 = compare_waic({
    "full_vote": mcmc_vote,
    "parsimonious_vote": mcmc_parsimonious,
    "no_vote": mcmc_novote
})

print(comparison_3)
comparison_3.to_csv('../parameter_estimates/waic_comparison_progressive_3models.csv')

# ---- SUMMARY ----
print("\n" + "="*60)
print("SUMMARY: Key vote parameters across models")
print("="*60)

print("\nFull vote model - vote parameters:")
print(summary_vote_df[summary_vote_df['Parameter'].str.contains('vote')].to_string())

print("\nParsimonious vote model - vote parameters:")
print(summary_parsimonious_df[summary_parsimonious_df['Parameter'].str.contains('vote')].to_string())

# Control analysis for Trust

In [ ]:
dataset = "trust"
data = pd.read_csv('../data/Ztable_trust_combined.csv')

# ---- EXCLUDE QUESTION 1 (politicians item) across all topics ----
print(f"Original data length: {len(data)}")
data = data[data['question'] != 1].reset_index(drop=True)
print(f"After excluding question == 1: {len(data)}")
print(data["model"].unique())

prompting = 0 
exclude_models = 0

print(f"\nWorking on dataset: {dataset}")
print(f"Prompting: {prompting}")
print("CONTROL ANALYSIS: Excluding question == 1 (politicians item)")

X, y, subject_ids, num_subjects, num_cutpoints, feature_names, unique_y = prepare_data(
    data, subset_fraction=0.10, use_subset=False, interaction_level=6,
    dataset=dataset, exclude_models=exclude_models, prompting=prompting)

# Plot heatmap for X
plt.figure(figsize=(8, 6))
sns.heatmap(X, cmap='viridis', cbar=False)
plt.xlabel("Predictors")
plt.ylabel("Observations")
plt.title(f"Design Matrix Heatmap - {dataset} (excl. question 1)")
plt.show()

# Calculate VIF
def calculate_vif(X):
    vif_data = pd.DataFrame()
    vif_data["feature"] = feature_names
    vif_data["VIF"] = [variance_inflation_factor(X, i) for i in range(X.shape[1])]
    return vif_data

vif_scores = calculate_vif(X)
print(vif_scores)

# Run inference
mcmc_full = run_inference(
    model=ordinal_model,
    X=X, y=y,
    subject_ids=subject_ids,
    num_subjects=num_subjects,
    num_cutpoints=num_cutpoints,
    rng_key=rng_key,
    num_warmup=num_warmup,
    num_samples=num_samples,
    num_chains=num_chains
)

az_data_full = az.from_numpyro(mcmc_full)

# Get summary
summary_full_df = summarize_results(mcmc_full, feature_names, focus_on_interactions=True)
summary_full_df = summary_full_df.sort_index()

# Add ROPE overlap
summary_full_df['rope_overlap'] = summary_full_df.apply(
    lambda row: calculate_overlap(
        rope_interval,
        row['HDI_2.5%'],
        row['HDI_97.5%']
    ),
    axis=1
)

print(summary_full_df)
mcmc_full.print_summary(exclude_deterministic=False)

# Save with distinct filename
summary_full_df.to_csv(
    f'../parameter_estimates/summary_full_GLM_{dataset}_prompting_{prompting}_excl_politicians.csv',
    index=False
)
plot_forestplot(
    summary_full_df, figsize=(10, 16), rope_interval=rope_interval,
    outcome=f"{dataset}_prompting_{prompting}_full_GLM_excl_politicians"
)

# Plot traceplot
az.plot_trace(az_data_full, var_names=["sigma_subject", "beta_scale", "cutpoints"])

# OLD Pipeline

## Full Model

In [ ]:
#make dataset
exclude_models = 0

#sycophancy or persuasion prompt
prompting = 0


if dataset == "misinfo" or dataset == "trust" or dataset == "private":
    X, y, subject_ids, num_subjects, num_cutpoints, feature_names, unique_y = prepare_data(
        data, subset_fraction=0.10, use_subset=False, interaction_level=6, dataset=dataset, exclude_models=exclude_models, prompting=prompting)
elif dataset == "extreme":
    X, y, subject_ids, num_subjects, num_cutpoints, feature_names, unique_y = prepare_data_extremism(
        data, subset_fraction=0.10, use_subset=False, interaction_level=6, exclude_models=exclude_models, prompting=prompting)
    
# # put X into dataframe
# X_df = pd.DataFrame(X)
# X_df.columns = feature_names
# X_df.head(10)

#plot heatmap for X
plt.figure(figsize=(8, 6))
sns.heatmap(X, cmap='viridis', cbar=False)
plt.xlabel("Predictors")
plt.ylabel("Observations")
plt.title("Design Matrix Heatmap")
plt.show()

# Calculate VIF for each predictor
def calculate_vif(X):
    vif_data = pd.DataFrame()
    vif_data["feature"] = feature_names
    vif_data["VIF"] = [variance_inflation_factor(X, i) for i in range(X.shape[1])]
    return vif_data

# Print VIF scores
vif_scores = calculate_vif(X)
print(vif_scores)


#set number of samples
num_samples = 2000
num_warmup = 2000

# Run inference 
if dataset == "misinfo" or dataset == "trust" or dataset == "private":
    mcmc_full = run_inference(
        model=ordinal_model, 
        X=X, 
        y=y, 
        subject_ids=subject_ids, 
        num_subjects=num_subjects, 
        num_cutpoints=num_cutpoints, 
        rng_key=rng_key,
        num_warmup=num_warmup, 
        num_samples=num_samples, 
        num_chains=num_chains
    )
elif dataset == "extreme":
    mcmc_full = run_inference(
            model=binomial_model, 
            X=X, 
            y=y, 
            subject_ids=subject_ids, 
            num_subjects=num_subjects, 
            num_cutpoints=None, 
            rng_key=rng_key,
            num_warmup=num_warmup, 
            num_samples=num_samples, 
            num_chains=num_chains
        )

az_data_full = az.from_numpyro(mcmc_full)

# Get summary focused on interactions
summary_full_df = summarize_results(mcmc_full, feature_names, focus_on_interactions=True)

#show summary_df ordered by index
summary_full_df = summary_full_df.sort_index()

# Add the overlap to the summary DataFrame
summary_full_df['rope_overlap'] = summary_full_df.apply(
    lambda row: calculate_overlap(
        rope_interval,
        row['HDI_2.5%'],
        row['HDI_97.5%']
    ),
    axis=1
)

summary_full_df


## Control model

In [ ]:
#make dataset
if dataset == "misinfo" or dataset == "trust" or dataset == "private":
    X, y, subject_ids, num_subjects, num_cutpoints, feature_names, unique_y = prepare_data(
        data, subset_fraction=0.10, use_subset=False, interaction_level=6, dataset=dataset, exclude_models=exclude_models, prompting=prompting)
elif dataset == "extreme":
    X, y, subject_ids, num_subjects, num_cutpoints, feature_names, unique_y = prepare_data_extremism(
        data, subset_fraction=0.10, use_subset=False, interaction_level=6, exclude_models=exclude_models, prompting=prompting)

# Specify the variables to exclude, everyting that contains 'iscontrol' and model variables
exclude_vars = ['istest',
                'model_gpt4o',
                'model_claude',
                'model_mistral',
                'ground_truth_researched_model_gpt4o',
                'ground_truth_researched_model_claude',
                'ground_truth_researched_model_mistral',
                'ground_truth_istest',
                'ground_truth_istest_model_gpt4o',
                'ground_truth_istest_model_claude',
                'ground_truth_istest_model_mistral',
                'researched_istest',
                'researched_istest_model_gpt4o',
                'researched_istest_model_claude',
                'researched_istest_model_mistral',
                'ground_truth_researched_istest',
                'ground_truth_researched_istest_model_gpt4o',
                'ground_truth_researched_istest_model_claude',
                'ground_truth_researched_istest_model_mistral',
                'time_istest',
                'time_model_gpt4o',
                'time_model_claude',
                'time_model_mistral',
                'time_ground_truth_model_gpt4o',
                'time_ground_truth_model_claude',
                'time_ground_truth_model_mistral',
                'time_researched_model_gpt4o',
                'time_researched_model_claude',
                'time_researched_model_mistral',
                'time_istest_model_gpt4o',
                'time_istest_model_claude',
                'time_istest_model_mistral',
                'time_ground_truth_researched_model_gpt4o',
                'time_ground_truth_researched_model_claude',
                'time_ground_truth_researched_model_mistral',
                'time_ground_truth_istest',
                'time_ground_truth_istest_model_gpt4o',
                'time_ground_truth_istest_model_claude',
                'time_ground_truth_istest_model_mistral',
                'time_researched_istest',
                'time_researched_istest_model_gpt4o',
                'time_researched_istest_model_claude',
                'time_researched_istest_model_mistral',
                'time_ground_truth_researched_istest',
                'time_ground_truth_researched_istest_model_gpt4o',
                'time_ground_truth_researched_istest_model_claude',
                'time_ground_truth_researched_istest_model_mistral',
                'model_sycophancy', 'model_persuasion', 
                'ground_truth_researched_model_sycophancy', 
                'ground_truth_researched_model_persuasion', 
                'ground_truth_istest_model_sycophancy', 
                'ground_truth_istest_model_persuasion',
                'researched_istest_model_sycophancy', 
                'researched_istest_model_persuasion', 
                'ground_truth_researched_istest_model_sycophancy', 
                'ground_truth_researched_istest_model_persuasion', 
                'time_model_sycophancy', 'time_model_persuasion', 
                'time_ground_truth_model_sycophancy', 
                'time_ground_truth_model_persuasion', 
                'time_researched_model_sycophancy', 
                'time_researched_model_persuasion', 
                'time_istest_model_sycophancy', 
                'time_istest_model_persuasion', 
                'time_ground_truth_researched_model_sycophancy', 
                'time_ground_truth_researched_model_persuasion', 
                'time_ground_truth_istest_model_sycophancy', 
                'time_ground_truth_istest_model_persuasion', 
                'time_researched_istest_model_sycophancy', 
                'time_researched_istest_model_persuasion', 
                'time_ground_truth_researched_istest_model_sycophancy', 
                'time_ground_truth_researched_istest_model_persuasion']


# Ensure the exclude_vars list is in the same format as feature_names
exclude_vars = [var for var in exclude_vars if var in feature_names]

# Create a boolean mask for the columns to keep
columns_to_keep = [var not in exclude_vars for var in feature_names]

# Filter X_array to exclude the specified columns
X_filtered = X[:, columns_to_keep]

# Filter feature_names to exclude the specified variables
feature_names_filtered = [var for var in feature_names if var not in exclude_vars]

# If needed, verify the shapes and contents
print(f"Original X shape: {X.shape}, Filtered X shape: {X_filtered.shape}")
print(f"Original feature names: {feature_names}")
print(f"Filtered feature names: {feature_names_filtered}")

#plot heatmap for X
plt.figure(figsize=(12, 8))
sns.heatmap(X_filtered, cmap='viridis', cbar=False)
plt.xlabel("Predictors")
plt.ylabel("Observations")
plt.title("Design Matrix Heatmap")
plt.show()


#set number of samples
num_samples = 2000
num_warmup = 2000

# Run inference 
if dataset == "misinfo" or dataset == "trust" or dataset == "private":
    mcmc_control = run_inference(
        model=ordinal_model, 
        X=X_filtered, 
        y=y, 
        subject_ids=subject_ids, 
        num_subjects=num_subjects, 
        num_cutpoints=num_cutpoints, 
        rng_key=rng_key,
        num_warmup=num_warmup, 
        num_samples=num_samples, 
        num_chains=num_chains
    )
elif dataset == "extreme":
    mcmc_control = run_inference(
            model=binomial_model, 
            X=X_filtered, 
            y=y, 
            subject_ids=subject_ids, 
            num_subjects=num_subjects, 
            num_cutpoints=None, 
            rng_key=rng_key,
            num_warmup=num_warmup, 
            num_samples=num_samples, 
            num_chains=num_chains
        )


#save model as netcdf
az_data_control = az.from_numpyro(mcmc_control)

# Get summary focused on interactions
summary_control_df = summarize_results(mcmc_control, feature_names_filtered, focus_on_interactions=True)
#show summary_df ordered by index
summary_control_df = summary_control_df.sort_index()

# Add the overlap to the summary DataFrame
summary_control_df['rope_overlap'] = summary_control_df.apply(
    lambda row: calculate_overlap(
        rope_interval,
        row['HDI_2.5%'],
        row['HDI_97.5%']
    ),
    axis=1
)

summary_control_df


#load full GLM (including model variables)
if dataset == "misinfo" and prompting == 0:
    az_data_full_with_model = az.from_netcdf(f"../GLM_fits/{dataset}_full_model.nc")
elif dataset == "misinfo" and prompting == 1:
    az_data_full_with_model = az.from_netcdf(f"../GLM_fits/{dataset}_prompting_{prompting}_full_model.nc")

#get waic for az_data_full_with_model
waic_model_full_with_model = az.waic(az_data_full_with_model)

# compare the two models using WAIC
waic_model_full = az.waic(mcmc_full)
waic_model_control = az.waic(mcmc_control)

if dataset == "misinfo":
    # Comparing WAIC values
    comparison = az.compare({"GLM_full_with_model": waic_model_full_with_model, "GLM_full_without_model": mcmc_full, "GLM_control": mcmc_control}, ic="waic")      
else:
    # Comparing WAIC values
    comparison = az.compare({"GLM_full": mcmc_full, "GLM_control": mcmc_control}, ic="waic")
print(comparison)

#plot the comparison
az.plot_compare(comparison)
#save plot
plt.savefig(f'../plots/WAIC_comparison_{dataset}_prompting_{prompting}.png', dpi=300, bbox_inches='tight')